In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

import torch
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments

import nltk
from nltk.corpus import stopwords
import re


In [19]:
import pandas as pd

df = pd.read_csv("reviews.csv")
print(df.head())
print(df.shape)


                                              Review
0  Everything is just amazing about this device. ...
1  Very powerful phone... Work fast.. camera qual...
2     Phone is super lightweighted and color is cool
3                            Heating problem persist
4                                       Good Quality
(8, 1)


In [18]:
df = df.rename(columns={"Review": "text"})
print(df.head())


                                                text     label
0  everything amazing device camera button works ...  POSITIVE
1  powerful phone work fast camera quality far go...  POSITIVE
2               phone super lightweighted color cool  POSITIVE
3                            heating problem persist  NEGATIVE
4                                       good quality  POSITIVE


In [17]:
len(df)


8

In [16]:
df["label"] = [
    "POSITIVE",
    "POSITIVE",
    "POSITIVE",
    "NEGATIVE",
    "POSITIVE",
    "NEGATIVE",
    "POSITIVE",
    "NEGATIVE"
]


In [6]:
print(df[["text", "label"]])


                                                text     label
0  Everything is just amazing about this device. ...  POSITIVE
1  Very powerful phone... Work fast.. camera qual...  POSITIVE
2     Phone is super lightweighted and color is cool  POSITIVE
3                            Heating problem persist  NEGATIVE
4                                       Good Quality  POSITIVE
5  I am writing to formally complain about my rec...  NEGATIVE
6                       A good phone at a good price  POSITIVE
7  Upgrading to the iPhone 16 was an absolute bre...  NEGATIVE


In [7]:
df["label"] = df["label"].map({
    "NEGATIVE": 0,
    "POSITIVE": 1
})

print(df[["text", "label"]])


                                                text  label
0  Everything is just amazing about this device. ...      1
1  Very powerful phone... Work fast.. camera qual...      1
2     Phone is super lightweighted and color is cool      1
3                            Heating problem persist      0
4                                       Good Quality      1
5  I am writing to formally complain about my rec...      0
6                       A good phone at a good price      1
7  Upgrading to the iPhone 16 was an absolute bre...      0


In [8]:
nltk.download("stopwords")
stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = text.lower()                      
    text = re.sub(r"[^a-zA-Z\s]", "", text)  
    text = " ".join(
        word for word in text.split() 
        if word not in stop_words
    )
    return text

df["text"] = df["text"].apply(clean_text)

print(df[["text", "label"]])


                                                text  label
0  everything amazing device camera button works ...      1
1  powerful phone work fast camera quality far go...      1
2               phone super lightweighted color cool      1
3                            heating problem persist      0
4                                       good quality      1
5  writing formally complain recent purchase ipho...      0
6                              good phone good price      1
7  upgrading iphone absolute breeze apple refined...      0


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\saksh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["text"],
    df["label"],
    test_size=0.25,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


Training samples: 6
Testing samples: 2


In [10]:
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)

train_encodings = tokenizer(
    list(X_train),
    truncation=True,
    padding=True
)

test_encodings = tokenizer(
    list(X_test),
    truncation=True,
    padding=True
)


In [11]:
train_encodings.keys()


dict_keys(['input_ids', 'attention_mask'])

In [12]:
import torch

class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = ReviewDataset(train_encodings, list(y_train))
test_dataset = ReviewDataset(test_encodings, list(y_test))


In [13]:
len(train_dataset), len(test_dataset)


(6, 2)

In [14]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [15]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    evaluation_strategy="epoch",
    logging_dir="./logs",
    logging_steps=1,
    save_strategy="no"
)


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    logging_dir="./logs",
    logging_steps=1,
    save_strategy="no"
)


In [ ]:
!pip install accelerate>=0.26.0


In [21]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    logging_dir="./logs",
    logging_steps=1,
    save_strategy="no"
)


In [22]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()


Step,Training Loss
1,0.716100
2,0.756700
3,0.658900
4,0.531300


TrainOutput(global_step=4, training_loss=0.665740355849266, metrics={'train_runtime': 8.0525, 'train_samples_per_second': 1.49, 'train_steps_per_second': 0.497, 'total_flos': 260795191104.0, 'train_loss': 0.665740355849266, 'epoch': 2.0})

In [23]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Get predictions on test set
predictions = trainer.predict(test_dataset)

# Convert model outputs to labels (0 or 1)
y_pred = np.argmax(predictions.predictions, axis=1)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_test, y_pred, average="binary"
)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)


Accuracy: 0.5
Precision: 0.5
Recall: 1.0
F1-score: 0.6666666666666666


In [24]:
# Predict sentiment for all reviews
all_encodings = tokenizer(
    list(df["text"]),
    truncation=True,
    padding=True,
    return_tensors="pt"
)

with torch.no_grad():
    outputs = model(**all_encodings)

df["Predicted_Sentiment"] = outputs.logits.argmax(dim=1)

# Convert back to readable labels
df["Predicted_Sentiment"] = df["Predicted_Sentiment"].map({
    0: "NEGATIVE",
    1: "POSITIVE"
})

print(df[["text", "Predicted_Sentiment"]])


KeyError: 'text'

In [25]:
print(df.columns)


Index(['Review'], dtype='object')


In [26]:
df = df.rename(columns={"Review": "text"})
print(df.columns)


Index(['text'], dtype='object')


In [27]:
import torch

all_encodings = tokenizer(
    list(df["text"]),
    truncation=True,
    padding=True,
    return_tensors="pt"
)

with torch.no_grad():
    outputs = model(**all_encodings)

df["Predicted_Sentiment"] = outputs.logits.argmax(dim=1)

df["Predicted_Sentiment"] = df["Predicted_Sentiment"].map({
    0: "NEGATIVE",
    1: "POSITIVE"
})

print(df[["text", "Predicted_Sentiment"]])


                                                text Predicted_Sentiment
0  Everything is just amazing about this device. ...            POSITIVE
1  Very powerful phone... Work fast.. camera qual...            POSITIVE
2     Phone is super lightweighted and color is cool            POSITIVE
3                            Heating problem persist            NEGATIVE
4                                       Good Quality            POSITIVE
5  I am writing to formally complain about my rec...            POSITIVE
6                       A good phone at a good price            POSITIVE
7  Upgrading to the iPhone 16 was an absolute bre...            POSITIVE


In [28]:
df.to_csv("final_sentiment_results.csv", index=False)
